# Monitoring a construction site with enhanced Sentinel-2 (hyper-ez + hyEnhance)

This notebook enhances Sentinel-2 from 10 m to 2 m with the **P2SR** method of
[hyEnhance](https://github.com/naikp13/hyEnhance) (Naik et al., 2025, *Remote Sensing* 17(11), 1878,
[doi:10.3390/rs17111878](https://doi.org/10.3390/rs17111878)), using a very-high-resolution RGB basemap as the guide
(**ESRI World Imagery**, **Google satellite imagery** or your own GeoTIFF), and uses the enhanced images to **track change over a site across several dates**.

| Role in `HSIEnhancer` | hyper-ez (hyperspectral workflow) | This notebook |
|---|---|---|
| low-resolution spectral image (`hsi_path`) | EnMAP 30 m (224 bands) | **Sentinel-2 L2A**, 12 bands on a 10 m grid, one per date |
| high-resolution guide (`msi_path`) | Sentinel-2 10 m, then ESRI 2 m | RGB basemap at 2 m (configurable): **ESRI World Imagery**, **Google satellite** (Map Tiles API) or your own GeoTIFF |

**Inputs**: an AOI polygon file (shapefile `.shp` with its `.shx`, `.dbf` and `.prj`, or a zipped shapefile, GeoPackage or
GeoJSON) and a list of dates.

**Workflow**

1. Load the AOI and build a UTM working grid around it.
2. For each date, find the closest Sentinel-2 L2A scene that covers the AOI and is cloud-free *over the AOI*
   (checked with the scene classification layer, SCL).
3. Read the 12 bands for each date (only the needed window), as surface reflectance on the 10 m grid.
4. Get one high-resolution RGB guide for the footprint (`GUIDE_SOURCE`: ESRI, Google or your own file).
5. Enhance every date to 2 m with hyEnhance `HSIEnhancer` and check that each result still matches its 10 m original.
6. Construction-related indices (NDVI, NDBI, BSI) over time on the enhanced images.
7. Sentinel-1 radar (VV and VH backscatter) for the same dates, from one orbit track, calibrated, aligned with Sentinel-2
   and speckle filtered. Radar sees through cloud and responds to structure and roughness rather than colour.
8. Change analysis that combines optical and radar evidence: change maps between dates, a "new structure" class from
   the radar, a confidence map (optical and radar agree, or only one sees the change), and changed area per interval.
9. Optional reduced-resolution check of the method (Wald protocol).

> **Important for monitoring.** The basemap (ESRI or Google) is a single mosaic with one acquisition date, and the same
> guide is used for every Sentinel-2 date. The 2 m edges therefore come from that one image, while the *spectral* values (and so
> the changes measured here) come from each Sentinel-2 date. A structure that is missing from the basemap still shows
> up as a change, but with softer, Sentinel-2-like edges. Read the change maps as "Sentinel-2 change, drawn on a 2 m
> grid", not as independent 2 m observations.

Runs on Google Colab or locally (Python 3.10+). Needs internet access to GitHub, PyPI, `earth-search.aws.element84.com`,
`sentinel-cogs.s3.us-west-2.amazonaws.com`, `sentinel-s1-l1c.s3.amazonaws.com` (Sentinel-1), and `services.arcgisonline.com` (ESRI) or `tile.googleapis.com` (Google).

## 1. Setup

In [ ]:
# Clone hyper-ez and hyEnhance next to this notebook (skipped if they already exist)
# and install any missing dependencies.
import importlib.util
import os
import subprocess
import sys

import shutil

HYENHANCE_BRANCH = os.environ.get("HYENHANCE_BRANCH", "main")

REPOS = {
    "hyper-ez": ("https://github.com/naikp13/hyper-ez.git", "main"),
    "hyEnhance": ("https://github.com/naikp13/hyEnhance.git", HYENHANCE_BRANCH),  # keep this folder name: hyper-ez imports `hyEnhance.src`
}
REPO_ROOT = os.environ.get("HYPEREZ_REPO_ROOT", os.getcwd())


def has_residual_fix(path):
    # hyEnhance versions before the patch-residual fix (Nov 2026) silently skipped every patch
    pp = os.path.join(path, "src", "patch_processor.py")
    return os.path.exists(pp) and "PatchFailure" in open(pp).read()


for name, (url, branch) in REPOS.items():
    path = os.path.join(REPO_ROOT, name)
    if name == "hyEnhance" and os.path.isdir(path) and not has_residual_fix(path):
        if "hyEnhance" in sys.modules:
            raise RuntimeError("An outdated hyEnhance is already imported. Restart the kernel "
                               "(Colab: Runtime > Restart session) and run the notebook from the top.")
        print("Replacing an outdated hyEnhance clone")
        shutil.rmtree(path)
    if not os.path.isdir(path):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", branch, url, path], check=True)

for p in (os.path.join(REPO_ROOT, "hyper-ez"), REPO_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

# module name -> pip package
REQUIRED = {
    "numpy": "numpy", "scipy": "scipy", "rasterio": "rasterio", "shapely": "shapely",
    "geopandas": "geopandas", "pyogrio": "pyogrio", "pandas": "pandas",
    "pystac_client": "pystac-client", "requests": "requests", "pytz": "pytz",
    "matplotlib": "matplotlib", "tqdm": "tqdm", "joblib": "joblib", "numba": "numba",
    "pywt": "PyWavelets", "sklearn": "scikit-learn", "skimage": "scikit-image",
    "PIL": "pillow",
}


OPENCV_PACKAGES = ["opencv-python", "opencv-python-headless", "opencv-contrib-python", "opencv-contrib-python-headless"]


def has_guided_filter():
    # Probe in a separate process so this kernel never caches a cv2 without the guided filter.
    # Check the function itself: some OpenCV builds (e.g. on Colab) have a `cv2.ximgproc` stub without it.
    probe = "import cv2, sys; sys.exit(0 if hasattr(getattr(cv2, 'ximgproc', None), 'guidedFilter') else 1)"
    return subprocess.run([sys.executable, "-c", probe], capture_output=True).returncode == 0


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)


missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", " ".join(missing))
    pip("install", "-q", *missing)

if not has_guided_filter():
    # The plain OpenCV wheels and the contrib wheels all install the same `cv2` module, and whichever is
    # installed last wins. Remove them all, then install only the contrib build.
    print("Installing OpenCV with contrib modules (needed for the guided filter)")
    pip("uninstall", "-y", "-q", *OPENCV_PACKAGES)
    pip("install", "-q", "opencv-contrib-python-headless")
    if not has_guided_filter():
        raise RuntimeError("cv2.ximgproc.guidedFilter is still missing after installing opencv-contrib-python-headless")
    if "cv2" in sys.modules:
        raise RuntimeError("OpenCV was replaced after it was imported. Restart the kernel "
                           "(Colab: Runtime > Restart session) and run the notebook from the top.")
    missing.append("opencv-contrib-python-headless")
if not missing:
    print("All dependencies are installed.")

In [ ]:
import json
import logging
import math
import re
import time
import warnings
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta

import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz
import rasterio
from pystac_client import Client
from rasterio.control import GroundControlPoint
from rasterio.coords import BoundingBox
from rasterio.crs import CRS
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.warp import Resampling, reproject, transform_bounds
from rasterio.windows import Window
from rasterio.windows import from_bounds as window_from_bounds
from scipy.ndimage import uniform_filter
from shapely.geometry import box, shape
from skimage.filters import threshold_otsu
from skimage.transform import resize

# hyper-ez: configuration and ESRI World Imagery download
from hyper_ez.config import STAC_API_URL, SENTINEL2_BAND_MAPPING, setup_matplotlib_style
from hyper_ez.image_processing import HyperspectralProcessor

# hyEnhance: the P2SR enhancement method
from hyEnhance.src import HSIEnhancer, HSIDataLoader, PatchProcessor

# Older hyEnhance versions silently skipped every patch of the sparse residual stage
if not hasattr(PatchProcessor(np.zeros((2, 2, 1)), np.zeros((2, 2, 1))), "stats"):
    raise ImportError("This hyEnhance version predates the patch-residual fix. Restart the kernel "
                      "(Colab: Runtime > Restart session) and run the notebook from the top; the setup cell "
                      "replaces the outdated copy.")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
setup_matplotlib_style()
plt.rcParams["font.size"] = 12

## 2. Parameters

- `AOI_PATH`: the construction-site polygon. For a shapefile, keep `.shp`, `.shx`, `.dbf` and `.prj` together in one
  folder and point to the `.shp` (a `.zip` of those files works too). Without a `.prj`, coordinates are assumed to be
  WGS84 longitude/latitude. Several polygons are merged into one AOI. On Colab, run the upload cell below first.
- `AOI_BUFFER_M`: margin around the AOI that is also processed. It gives context in the maps and a stable surrounding
  area for the change threshold.
- `DATES`: the dates to monitor. For each one the closest suitable scene within `SEARCH_WINDOW_DAYS` is used.
  Prefer dates from the same season, because crop and grass cycles also change NDVI.
- `MIN_AOI_CLEAR`: the share of AOI pixels that must be free of cloud, cloud shadow and cirrus according to SCL.
- Enhancement parameters: `RESIDUAL_WEIGHT = 0.5`, `DETAIL_WEIGHT = 0.25` and `GUIDE_RADIUS = 4` scored best for
  Sentinel-2 in reduced-resolution checks on four scenes (section 10). Against the same settings with the residual
  switched off, they lowered RMSE by 4–6 %, SAM by 2–4 % and ERGAS by 5 % on three scenes, and made no measurable
  difference on the fourth. hyper-ez uses `GUIDE_RADIUS = 10` and
  `DETAIL_WEIGHT = 1` for hyperspectral data.
- `STRIDE = 2` processes a quarter of the patches of `STRIDE = 1`. For the default ~190 × 200 grid the patch stage
  takes about 2 minutes per date on a 4-core workstation, but about 16 minutes per date on free Colab. `STRIDE = 4`
  processes about a quarter as many patches again; the patches overlap less, so check the result visually.
- `GUIDE_SOURCE` picks the high-resolution guide (details in section 5):
  - `"esri"`: ESRI World Imagery, no key needed. The export service returns at most 4096 pixels per side, so the AOI
    plus buffer can be up to about 8 km across at 2 m.
  - `"google"`: Google satellite imagery (the imagery behind Google Maps / Google Earth) through the official
    Google Maps Platform **Map Tiles API**. It needs your own API key with the Map Tiles API enabled; see section 5 for
    the key and the terms of use.
  - `"file"`: any georeferenced RGB GeoTIFF you provide in `GUIDE_FILE`, for example a drone orthomosaic or imagery
    you are licensed to use. It is reprojected onto the working grid.

- Sentinel-1 radar (section 8): `USE_S1 = False` runs the optical-only analysis.
  - `S1_WINDOW_DAYS`: all acquisitions of the chosen track within this many days of each Sentinel-2 scene are averaged,
    which reduces speckle. Sentinel-1 revisits a track every 12 days with one satellite and every 6 days with two
    (2016–2021 and again from 2025), so 6 days gives one to three acquisitions per date.
  - `S1_ORBIT`: the relative orbit (track) to use. `None` picks the track with the most acquisitions that covers every
    date. Only one track is used, because backscatter depends on the viewing angle.
  - `S1_CHANGE_DB`: the backscatter change, in dB, that counts as radar change on its own. `S1_WEIGHT` sets how much the radar
    counts against the optical change score (0 = optical only, 0.5 = equal weight).

The defaults monitor the Tesla Gigafactory Berlin-Brandenburg site (Grünheide, Germany), which was built in 2020 and
2021, using the sample AOI written by the next cell.

In [ ]:
AOI_PATH = "aoi/construction_site.shp"
AOI_BUFFER_M = 150

DATES = ["2020-03-25", "2020-06-25", "2020-09-20", "2021-04-20"]
SEARCH_WINDOW_DAYS = 25      # +/- days around each date
MAX_SCENE_CLOUD = 60         # scene-level cloud cover pre-filter, percent
MIN_AOI_CLEAR = 0.95         # required clear fraction inside the AOI

# Grids
S2_RES_M = 10.0
GUIDE_RES_M = 2.0

# High-resolution guide: "esri", "google" or "file"
GUIDE_SOURCE = "esri"
GUIDE_FILE = None              # GeoTIFF path, only for GUIDE_SOURCE = "file"
GOOGLE_MAX_ZOOM = 19           # cap on the Google tile zoom level (zoom 19 is about 0.2 m per pixel)

# P2SR / HSIEnhancer parameters
PATCH_SIZE = 12
STRIDE = 2
RESIDUAL_WEIGHT = 0.5  # weight of the patch-wise sparse residual; 0 switches the stage off
DETAIL_WEIGHT = 0.25   # hyper-ez default: 1
GUIDE_RADIUS = 4       # hyper-ez default: 10

# Change analysis
NDVI_CHANGE = 0.15        # |dNDVI| above this counts as vegetation loss / gain
CVA_THRESHOLD = None      # change-vector magnitude threshold (reflectance); None = automatic (Otsu, see section 9)

# Sentinel-1 radar
USE_S1 = True
S1_WINDOW_DAYS = 6        # average the chosen track's acquisitions within +/- days of each Sentinel-2 scene
S1_ORBIT = None           # relative orbit (track) number; None = the best track that covers every date
S1_LEE_SIZE = 5           # speckle filter window, 10 m pixels
S1_CHANGE_DB = 3.0        # backscatter change (dB) that counts as radar change on its own
S1_WEIGHT = 0.5           # weight of radar in the combined change score; 0 = Sentinel-2 only

S2_BANDS = list(SENTINEL2_BAND_MAPPING.keys())   # B01..B12 with B8A after B08
OUT_DIR = "s2_monitoring_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SCALE = S2_RES_M / GUIDE_RES_M
assert abs(SCALE - round(SCALE)) < 1e-9, "GUIDE_RES_M must divide 10 m evenly"
SCALE = int(round(SCALE))
print(f"Scale factor: {SCALE}x  ({S2_RES_M:g} m -> {GUIDE_RES_M:g} m), {len(DATES)} dates")

In [ ]:
# Sample AOI (Tesla Gigafactory Berlin-Brandenburg construction site). Skipped when AOI_PATH already exists.
# Replace AOI_PATH above with your own file, or on Colab upload it with the next cell.
from shapely.geometry import Polygon

if not os.path.exists(AOI_PATH):
    os.makedirs(os.path.dirname(AOI_PATH) or ".", exist_ok=True)
    site = Polygon([(13.7855, 52.3905), (13.8065, 52.3905), (13.8095, 52.3985),
                    (13.8030, 52.4035), (13.7855, 52.4020), (13.7855, 52.3905)])
    gpd.GeoDataFrame({"name": ["Gigafactory Berlin-Brandenburg"]}, geometry=[site], crs="EPSG:4326").to_file(AOI_PATH)
    print("Wrote sample AOI:", AOI_PATH)

In [ ]:
# Colab only: upload your own AOI (select the .shp, .shx, .dbf and .prj together, or one .zip).
# Leave UPLOAD_AOI = False to keep AOI_PATH.
UPLOAD_AOI = False

if UPLOAD_AOI:
    from google.colab import files
    os.makedirs("aoi_upload", exist_ok=True)
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join("aoi_upload", name), "wb") as fh:
            fh.write(data)
    main = [n for n in uploaded if n.lower().endswith((".shp", ".zip", ".gpkg", ".geojson", ".json"))]
    assert main, "Upload a .shp (with .shx, .dbf, .prj), a .zip, a .gpkg or a .geojson"
    AOI_PATH = os.path.join("aoi_upload", main[0])
    print("AOI_PATH =", AOI_PATH)

## 3. Load the AOI and build the working grid

In [ ]:
def load_aoi(path):
    gdf = gpd.read_file(path)
    if gdf.empty:
        raise ValueError(f"{path} has no features")
    if gdf.crs is None:
        print("No .prj / CRS found: assuming WGS84 longitude/latitude (EPSG:4326).")
        gdf = gdf.set_crs(4326)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    if not gdf.geom_type.isin(["Polygon", "MultiPolygon"]).all():
        raise ValueError("The AOI must contain polygons")
    return gdf


def snapped_bounds(geom, buffer_m, snap=60.0):
    # Bounds of the buffered AOI, snapped outward to multiples of 60 m, so the 10/20/60 m Sentinel-2
    # pixels and the ESRI pixels all nest exactly
    minx, miny, maxx, maxy = geom.buffer(buffer_m).bounds
    return BoundingBox(math.floor(minx / snap) * snap, math.floor(miny / snap) * snap,
                       math.ceil(maxx / snap) * snap, math.ceil(maxy / snap) * snap)


aoi_gdf = load_aoi(AOI_PATH)
work_crs = CRS.from_user_input(aoi_gdf.estimate_utm_crs())
aoi_utm = aoi_gdf.to_crs(work_crs).geometry.union_all() if hasattr(aoi_gdf.geometry, "union_all") \
    else aoi_gdf.to_crs(work_crs).unary_union
aoi_wgs84 = gpd.GeoSeries([aoi_utm], crs=work_crs).to_crs(4326).iloc[0]
work_bounds = snapped_bounds(aoi_utm, AOI_BUFFER_M)

n_rows = int(round((work_bounds.top - work_bounds.bottom) / S2_RES_M))
n_cols = int(round((work_bounds.right - work_bounds.left) / S2_RES_M))
if GUIDE_SOURCE == "esri":
    assert max(n_rows, n_cols) * SCALE <= 4096, "AOI + buffer is too large for one ESRI export: shrink it or raise GUIDE_RES_M"


def aoi_mask(res):
    # True inside the AOI on a grid of pixel size `res`
    h = int(round((work_bounds.top - work_bounds.bottom) / res))
    w = int(round((work_bounds.right - work_bounds.left) / res))
    return geometry_mask([aoi_utm], out_shape=(h, w), transform=transform_from_bounds(*work_bounds, w, h), invert=True)


def aoi_outline(ax, res, color="yellow"):
    geoms = aoi_utm.geoms if aoi_utm.geom_type == "MultiPolygon" else [aoi_utm]
    for g in geoms:
        x, y = g.exterior.xy
        ax.plot((np.array(x) - work_bounds.left) / res - 0.5, (work_bounds.top - np.array(y)) / res - 0.5,
                color=color, lw=1.5)


print(f"AOI: {len(aoi_gdf)} feature(s), {aoi_utm.area / 1e4:.1f} ha, working CRS {work_crs}")
print(f"Grid: {n_rows} x {n_cols} px at {S2_RES_M:g} m  ->  {n_rows * SCALE} x {n_cols * SCALE} px at {GUIDE_RES_M:g} m")

## 4. Choose one Sentinel-2 scene per date

Candidates come from the AWS Earth Search STAC API: scenes within `SEARCH_WINDOW_DAYS` of the date whose footprint covers
the AOI and whose scene-level cloud cover is below `MAX_SCENE_CLOUD`. They are tried from the closest date outward, and
the first one whose SCL layer marks at least `MIN_AOI_CLEAR` of the AOI as clear is kept. If none passes, the clearest
candidate is kept and flagged.

SCL classes treated as not clear: 0 no data, 1 saturated, 3 cloud shadow, 8 and 9 cloud, 10 cirrus. Very bright new
roofs or concrete are sometimes classed as cloud; look at the true-colour images in section 6 if a date seems to lose
many pixels.

In [ ]:
NOT_CLEAR_SCL = [0, 1, 3, 8, 9, 10]
GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", AWS_NO_SIGN_REQUEST="YES",
                CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif")


def read_asset(href, res, resampling, src_nodata=0):
    # Read only the window of a COG that covers the working grid and resample it onto the grid
    h = int(round((work_bounds.top - work_bounds.bottom) / res))
    w = int(round((work_bounds.right - work_bounds.left) / res))
    out = np.full((h, w), np.nan, dtype=np.float32)
    with rasterio.Env(**GDAL_ENV), rasterio.open(href) as src:
        # the grid's corners in the source CRS (they differ when the scene is in a neighbouring UTM zone)
        l, b, r, t = transform_bounds(work_crs, src.crs, *work_bounds)
        pad = 2 * max(src.res)
        win = window_from_bounds(l - pad, b - pad, r + pad, t + pad, transform=src.transform)
        win = win.round_offsets().round_lengths()
        data = src.read(1, window=win).astype(np.float32)
        data[data == src_nodata] = np.nan
        reproject(data, out, src_transform=src.window_transform(win), src_crs=src.crs, src_nodata=np.nan,
                  dst_transform=transform_from_bounds(*work_bounds, w, h), dst_crs=work_crs,
                  dst_nodata=np.nan, resampling=resampling)
    return out


def aoi_clear_fraction(item):
    scl = read_asset(item.assets["scl"].href, S2_RES_M, Resampling.nearest, src_nodata=-1)
    clear = np.isfinite(scl) & ~np.isin(scl, NOT_CLEAR_SCL)
    return clear[AOI_MASK_10].mean(), clear


def stac_candidates(date):
    t0 = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=pytz.UTC)
    rng = f"{(t0 - timedelta(days=SEARCH_WINDOW_DAYS)):%Y-%m-%d}/{(t0 + timedelta(days=SEARCH_WINDOW_DAYS)):%Y-%m-%d}"
    items = Client.open(STAC_API_URL).search(
        collections=["sentinel-2-l2a"], intersects=aoi_wgs84.__geo_interface__, datetime=rng,
        query={"eo:cloud_cover": {"lt": MAX_SCENE_CLOUD}}).items()
    items = [it for it in items if shape(it.geometry).contains(aoi_wgs84)]
    return sorted(items, key=lambda it: abs(it.datetime - t0))


def select_scene(date):
    best = None
    for item in stac_candidates(date):
        frac, clear = aoi_clear_fraction(item)
        if best is None or frac > best[1]:
            best = (item, frac, clear)
        if frac >= MIN_AOI_CLEAR:
            break
    if best is None:
        raise RuntimeError(f"No Sentinel-2 scene covers the AOI within {SEARCH_WINDOW_DAYS} days of {date}")
    return best


AOI_MASK_10 = aoi_mask(S2_RES_M)
scenes = []
for d in DATES:
    item, frac, clear = select_scene(d)
    flag = "" if frac >= MIN_AOI_CLEAR else "  <- below MIN_AOI_CLEAR, cloudy pixels are masked"
    print(f"{d}: {item.id}  ({item.datetime:%Y-%m-%d}, AOI clear {frac:.0%}){flag}")
    scenes.append(dict(requested=d, item=item, date=item.datetime.date(), clear_frac=frac, clear_10=clear))

## 5. Read Sentinel-2 for every date and get the high-resolution guide

Digital numbers are converted to surface reflectance with the scale and offset in the STAC metadata. Scenes from
processing baseline 04.00 onward carry a +1000 offset; it is removed unless Earth Search has already removed it
(`earthsearch:boa_offset_applied`). The 20 m and 60 m bands are resampled bilinearly to 10 m.

The guide is fetched once and used for every date, according to `GUIDE_SOURCE`:

- **`"esri"`** uses hyper-ez's `HyperspectralProcessor.acquire_high_resolution_imagery`.
- **`"google"`** uses the Google Maps Platform [Map Tiles API](https://developers.google.com/maps/documentation/tile/2d-tiles-overview)
  (`mapType: satellite`). The notebook picks the zoom level whose pixel size is at or below `GUIDE_RES_M` at the AOI's
  latitude, downloads the tiles, mosaics them in Web Mercator and reprojects the mosaic onto the working grid. The
  attribution text Google returns for the area is printed and saved in the GeoTIFF metadata.
  - **API key**: create a Google Cloud project with billing, enable the *Map Tiles API*, and create an API key. On Colab,
    store it as a secret named `GOOGLE_MAPS_API_KEY` (key icon in the left sidebar), or set the environment variable of
    the same name. Do not paste the key into the notebook.
  - **Cost**: a 3 km × 3 km AOI at 2 m needs on the order of 100 tiles (zoom 16), which falls within the API's monthly
    free usage at the time of writing; check current pricing.
  - **Terms of use**: Google Maps Platform terms restrict using Google imagery to create new data, such as image analysis
    or derived products, and they require attribution. An enhanced Sentinel-2 product built from Google imagery may
    count as derived content. Check the current terms, or get permission from Google, before using this option beyond
    personal experimentation, and do not redistribute the outputs.
- **`"file"`** reads `GUIDE_FILE` (any CRS; the first three bands are taken as R, G, B and stretched to 8 bit if needed).

In [ ]:
def band_scale_offset(asset, item):
    # Earth Search flags scenes whose BOA offset it already removed; applying -0.1 again would make dark pixels negative
    rb = (asset.extra_fields.get("raster:bands") or [{}])[0]
    scale = rb.get("scale", 1e-4)
    if item.properties.get("earthsearch:boa_offset_applied", False):
        return scale, 0.0
    if "offset" in rb:
        return scale, rb["offset"]
    baseline = float(item.properties.get("s2:processing_baseline", "0") or 0)
    return scale, (-0.1 if baseline >= 4.0 else 0.0)


def read_s2_reflectance(item):
    layers = []
    for b in S2_BANDS:
        asset = item.assets[SENTINEL2_BAND_MAPPING[b]]
        scale, offset = band_scale_offset(asset, item)
        layers.append(read_asset(asset.href, S2_RES_M, Resampling.bilinear) * scale + offset)
    return np.stack(layers, axis=-1).astype(np.float32)


def write_geotiff(path, data_hwc, band_names=None, dtype="float32"):
    h, w, c = data_hwc.shape
    profile = dict(driver="GTiff", height=h, width=w, count=c, dtype=dtype, crs=work_crs,
                   transform=transform_from_bounds(*work_bounds, w, h), compress="deflate",
                   tiled=True, blockxsize=256, blockysize=256)
    if dtype == "float32":
        profile["nodata"] = np.nan
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(np.moveaxis(data_hwc, -1, 0).astype(dtype))
        for i, name in enumerate(band_names or [], start=1):
            dst.set_band_description(i, name)


for sc in scenes:
    sc["s2"] = read_s2_reflectance(sc["item"])
    sc["s2_path"] = os.path.join(OUT_DIR, f"sentinel2_10m_{sc['date']}.tif")
    write_geotiff(sc["s2_path"], sc["s2"], S2_BANDS)
    print(f"{sc['date']}: {sc['s2'].shape} -> {sc['s2_path']}")

# ---- High-resolution guide (one image for all dates) ----
GRID_H, GRID_W = n_rows * SCALE, n_cols * SCALE
GRID_TRANSFORM = transform_from_bounds(*work_bounds, GRID_W, GRID_H)
WEB_MERCATOR = CRS.from_epsg(3857)
WORLD_M = 20037508.342789244  # half the Web Mercator world width, metres


def to_grid(rgb_chw, src_transform, src_crs):
    # Reproject a (3, rows, cols) uint8 image onto the 2 m working grid
    out = np.zeros((3, GRID_H, GRID_W), dtype=np.uint8)
    reproject(rgb_chw, out, src_transform=src_transform, src_crs=src_crs, src_nodata=0,
              dst_transform=GRID_TRANSFORM, dst_crs=work_crs, dst_nodata=0, resampling=Resampling.bilinear)
    return np.moveaxis(out, 0, -1)


def guide_from_esri():
    downloaded = HyperspectralProcessor().acquire_high_resolution_imagery(work_bounds, work_crs,
                                                                         target_resolution=GUIDE_RES_M)
    with rasterio.open(downloaded) as src:
        rgb = np.moveaxis(src.read(), 0, -1)[..., :3]  # keep R, G, B
    os.remove(downloaded)
    return rgb, "Esri, Maxar, Earthstar Geographics, and the GIS User Community"


def google_api_key():
    key = os.environ.get("GOOGLE_MAPS_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GOOGLE_MAPS_API_KEY")
        except Exception:
            key = None
    if not key:
        raise RuntimeError("Set a GOOGLE_MAPS_API_KEY Colab secret or environment variable (Map Tiles API enabled).")
    return key


def google_check(resp, what):
    # Raise with Google's own error message, never echoing the request URL (it could carry the key)
    if resp.ok:
        return
    try:
        err = resp.json().get("error", {})
        detail = f"{err.get('status', '')}: {err.get('message', '')}"
    except Exception:
        detail = resp.text[:300]
    hint = ""
    if resp.status_code == 403:
        hint = ("\nCheck in Google Cloud that, in the project the key belongs to, the Map Tiles API is enabled, "
                "billing is linked, and the key's API restrictions (if any) include the Map Tiles API.")
    raise RuntimeError(f"Google {what} failed ({resp.status_code}) {detail}{hint}") from None


def guide_from_google():
    import requests
    from io import BytesIO
    from PIL import Image

    key = google_api_key()
    api = "https://tile.googleapis.com"
    auth = {"X-Goog-Api-Key": key}   # header, so the key never appears in URLs or tracebacks
    session = requests.post(f"{api}/v1/createSession", headers=auth, timeout=60,
                            json={"mapType": "satellite", "language": "en-US", "region": "US"})
    google_check(session, "createSession")
    token = session.json()["session"]

    # Zoom level whose ground pixel size is at or below GUIDE_RES_M at the AOI's latitude
    lat = aoi_wgs84.centroid.y
    zoom = math.ceil(math.log2(2 * WORLD_M * math.cos(math.radians(lat)) / (256 * GUIDE_RES_M)))
    zoom = int(min(max(zoom, 1), GOOGLE_MAX_ZOOM))
    tile_m = 2 * WORLD_M / 2 ** zoom
    l, b, r, t = transform_bounds(work_crs, WEB_MERCATOR, *work_bounds)
    x0, x1 = int((l + WORLD_M) // tile_m), int((r + WORLD_M) // tile_m)
    y0, y1 = int((WORLD_M - t) // tile_m), int((WORLD_M - b) // tile_m)
    n_tiles = (x1 - x0 + 1) * (y1 - y0 + 1)
    print(f"Google satellite: zoom {zoom} ({tile_m / 256 * math.cos(math.radians(lat)):.2f} m/px on the ground), "
          f"{n_tiles} tiles")
    if n_tiles > 2000:
        raise RuntimeError(f"{n_tiles} tiles is a lot of API usage: shrink the AOI or raise GUIDE_RES_M")

    mosaic = np.zeros((3, (y1 - y0 + 1) * 256, (x1 - x0 + 1) * 256), dtype=np.uint8)
    with requests.Session() as http:
        http.headers.update(auth)
        for ty in range(y0, y1 + 1):
            for tx in range(x0, x1 + 1):
                resp = http.get(f"{api}/v1/2dtiles/{zoom}/{tx}/{ty}", params={"session": token}, timeout=60)
                google_check(resp, f"tile {zoom}/{tx}/{ty}")
                tile = np.asarray(Image.open(BytesIO(resp.content)).convert("RGB").resize((256, 256)))
                r0, c0 = (ty - y0) * 256, (tx - x0) * 256
                mosaic[:, r0:r0 + 256, c0:c0 + 256] = np.moveaxis(tile, -1, 0)
    mosaic[mosaic == 0] = 1  # keep true black pixels from being read as no-data
    mosaic_transform = rasterio.transform.from_origin(-WORLD_M + x0 * tile_m, WORLD_M - y0 * tile_m,
                                                      tile_m / 256, tile_m / 256)

    # Attribution Google requires for this area (best effort)
    attribution = "Google"
    try:
        north, south = aoi_wgs84.bounds[3], aoi_wgs84.bounds[1]
        east, west = aoi_wgs84.bounds[2], aoi_wgs84.bounds[0]
        vp = requests.get(f"{api}/tile/v1/viewport", timeout=60, headers=auth, params={
            "session": token, "zoom": zoom, "north": north, "south": south, "east": east, "west": west})
        attribution = vp.json().get("copyright") or attribution
    except Exception:
        pass
    return to_grid(mosaic, mosaic_transform, WEB_MERCATOR), attribution


def guide_from_file(path):
    with rasterio.open(path) as src:
        if src.count < 3:
            raise ValueError(f"{path} has {src.count} band(s); an RGB image (3+ bands) is needed")
        data = src.read([1, 2, 3]).astype(np.float32)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
        if src.dtypes[0] != "uint8":  # stretch each band to 1..255
            lo = np.nanpercentile(data, 0.5, axis=(1, 2), keepdims=True)
            hi = np.nanpercentile(data, 99.5, axis=(1, 2), keepdims=True)
            data = np.clip((data - lo) / (hi - lo + 1e-12), 0, 1) * 254 + 1
        data = np.nan_to_num(data, nan=0).astype(np.uint8)
        return to_grid(data, src.transform, src.crs), os.path.basename(path)


if GUIDE_SOURCE == "esri":
    guide, guide_attribution = guide_from_esri()
elif GUIDE_SOURCE == "google":
    guide, guide_attribution = guide_from_google()
elif GUIDE_SOURCE == "file":
    assert GUIDE_FILE, "Set GUIDE_FILE to a GeoTIFF path"
    guide, guide_attribution = guide_from_file(GUIDE_FILE)
else:
    raise ValueError(f"Unknown GUIDE_SOURCE {GUIDE_SOURCE!r}: use 'esri', 'google' or 'file'")

assert guide.shape[:2] == (GRID_H, GRID_W), f"Grid mismatch: guide {guide.shape[:2]}"
empty = (guide == 0).all(-1).mean()
if empty > 0.01:
    print(f"Warning: {empty:.0%} of the guide has no data (outside the imagery's coverage)")
GUIDE_LABEL = {"esri": "ESRI", "google": "Google", "file": "Guide"}[GUIDE_SOURCE]
guide_path = os.path.join(OUT_DIR, f"guide_{GUIDE_SOURCE}_{GUIDE_RES_M:g}m.tif")
write_geotiff(guide_path, guide, ["Red", "Green", "Blue"], dtype="uint8")
with rasterio.open(guide_path, "r+") as dst:
    dst.update_tags(SOURCE=GUIDE_SOURCE, ATTRIBUTION=guide_attribution)
print(f"{GUIDE_LABEL} guide: {guide.shape} -> {guide_path}   (imagery: {guide_attribution})")

## 6. Enhance every date to 2 m

`HSIEnhancer(guide, sentinel2)` runs the three P2SR stages:

1. **Patch-wise sparse residual** (`PatchProcessor`). For each 12 × 12 Sentinel-2 patch, wavelet, FastICA and NMF
   components are extracted. A joint dictionary is learned over [guide degraded to 10 m | components]. The guide's
   rows code the full-resolution guide, and the component rows turn those codes into 2 m component estimates. The
   difference from the upsampled 10 m estimate is mapped back to the 12 bands with a ridge regression fitted on the patch.
2. **Detail injection** (`HSIUpsampler`): the guide's high-frequency detail is added, then each band's mean and standard
   deviation are restored.
3. **Guided filter** per band, with the high-resolution image as guide.

The patch stage reports how many patches succeeded. It raises an error if every patch fails, instead of quietly
returning a zero residual.

After each run the 2 m result is averaged back to 10 m and compared with its original (SAM in degrees, ERGAS; lower is
better for both, and ERGAS under about 3 is usually considered good). This confirms the enhancement kept each date's
spectra, which the change analysis relies on.

In [ ]:
def block_mean(cube, f):
    h, w, c = cube.shape
    return cube[: h // f * f, : w // f * f].reshape(h // f, f, w // f, f, c).mean(axis=(1, 3))


def quality_metrics(ref, est, ratio):
    valid = np.isfinite(ref).all(-1) & np.isfinite(est).all(-1)
    r, e = ref[valid].astype(np.float64), est[valid].astype(np.float64)
    rmse_b = np.sqrt(((r - e) ** 2).mean(0))
    cc_b = np.array([np.corrcoef(r[:, i], e[:, i])[0, 1] for i in range(r.shape[1])])
    cos = (r * e).sum(1) / (np.linalg.norm(r, axis=1) * np.linalg.norm(e, axis=1) + 1e-12)
    return dict(RMSE=np.sqrt((rmse_b ** 2).mean()), PSNR=20 * np.log10(r.max() / (np.sqrt((rmse_b ** 2).mean()) + 1e-12)),
                SAM=np.degrees(np.arccos(np.clip(cos, -1, 1))).mean(),
                ERGAS=100.0 / ratio * np.sqrt(np.mean((rmse_b / (np.abs(r.mean(0)) + 1e-12)) ** 2)), CC=cc_b.mean())


def enhance(guide_path, s2_path):
    enhancer = HSIEnhancer(guide_path, s2_path)
    kept = np.isnan(HSIDataLoader.load_image(s2_path)).mean(axis=(0, 1)) <= 0.5
    if not kept.all():
        raise RuntimeError(f"{s2_path}: bands {np.array(S2_BANDS)[~kept].tolist()} are mostly no-data")
    out = enhancer.fuse_to_enhance(patch_size=PATCH_SIZE, stride=STRIDE, guide_radius=GUIDE_RADIUS,
                                   detail_weight=DETAIL_WEIGHT, residual_weight=RESIDUAL_WEIGHT)
    return out.astype(np.float32), enhancer


for sc in scenes:
    sc["enh"], enhancer = enhance(guide_path, sc["s2_path"])
    sc["enh_path"] = os.path.join(OUT_DIR, f"sentinel2_enhanced_{GUIDE_RES_M:g}m_{sc['date']}.tif")
    write_geotiff(sc["enh_path"], sc["enh"], S2_BANDS)
    sc["consistency"] = quality_metrics(sc["s2"], block_mean(sc["enh"], SCALE), SCALE)
    q, ps = sc["consistency"], getattr(enhancer, "patch_stats", {}) or {}
    patches = (f"patches ok {ps['ok']}, failed {ps['failed']}" if ps else "patch stage off")
    print(f"{sc['date']}: {sc['enh'].shape} -> {sc['enh_path']}   {patches}   "
          f"SAM {q['SAM']:.2f} deg, ERGAS {q['ERGAS']:.2f}")
    if ps.get("failed"):
        print("   first patch error:", ps["first_error"])

### Images over time

In [ ]:
def stretch(rgb, lo_hi):
    lo, hi = lo_hi
    return np.clip((rgb - lo) / (hi - lo + 1e-12), 0, 1)


def bands(cube, names):
    return cube[..., [S2_BANDS.index(n) for n in names]]


TRUE_COLOR = ["B04", "B03", "B02"]
# One stretch for all dates (from the first date), so brightness differences between dates are real
ref_rgb = bands(scenes[0]["s2"], TRUE_COLOR)
LIMITS = (np.nanpercentile(ref_rgb, 2, axis=(0, 1)), np.nanpercentile(ref_rgb, 98, axis=(0, 1)))

n = len(scenes)
fig, axes = plt.subplots(2, n + 1, figsize=(4.6 * (n + 1), 9.4))
for j, sc in enumerate(scenes):
    lr = np.repeat(np.repeat(bands(sc["s2"], TRUE_COLOR), SCALE, 0), SCALE, 1)
    axes[0, j].imshow(stretch(lr, LIMITS), interpolation="nearest")
    axes[1, j].imshow(stretch(bands(sc["enh"], TRUE_COLOR), LIMITS), interpolation="nearest")
    axes[0, j].set_title(f"{sc['date']}\nSentinel-2 {S2_RES_M:g} m")
    axes[1, j].set_title(f"Enhanced {GUIDE_RES_M:g} m")
axes[0, n].imshow(guide); axes[0, n].set_title(f"{GUIDE_LABEL} guide {GUIDE_RES_M:g} m\n(single date)")
axes[1, n].text(0.02, 0.98, f"Guide imagery:\n{guide_attribution}", va="top", fontsize=9, wrap=True,
                transform=axes[1, n].transAxes)
axes[1, n].axis("off")
for ax in axes.ravel():
    if ax.has_data():
        aoi_outline(ax, GUIDE_RES_M)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 7. Construction indices over time

| Index | Formula (Sentinel-2 bands) | What it shows on a construction site |
|---|---|---|
| NDVI | (B08 − B04) / (B08 + B04) | vegetation; drops when the site is cleared |
| BSI (bare soil index) | ((B11 + B04) − (B08 + B02)) / ((B11 + B04) + (B08 + B02)) | exposed soil and earthworks; rises after clearing |
| NDBI (built-up index) | (B11 − B08) / (B11 + B08) | built surfaces such as concrete and roofs; rises as structures appear |

B11 is a 20 m band, so BSI and NDBI carry less true spatial detail than NDVI. Statistics use only clear pixels inside the AOI.

In [ ]:
def indices(cube):
    b = {n: cube[..., S2_BANDS.index(n)] for n in ("B02", "B04", "B08", "B11")}
    eps = 1e-6
    return {
        "NDVI": (b["B08"] - b["B04"]) / (b["B08"] + b["B04"] + eps),
        "BSI": ((b["B11"] + b["B04"]) - (b["B08"] + b["B02"])) / ((b["B11"] + b["B04"]) + (b["B08"] + b["B02"]) + eps),
        "NDBI": (b["B11"] - b["B08"]) / (b["B11"] + b["B08"] + eps),
    }


AOI_MASK_HR = aoi_mask(GUIDE_RES_M)
for sc in scenes:
    sc["clear_hr"] = np.repeat(np.repeat(sc["clear_10"], SCALE, 0), SCALE, 1)
    sc["idx"] = indices(sc["enh"])
    sc["idx_10"] = indices(sc["s2"])
    sel = AOI_MASK_HR & sc["clear_hr"]
    sc["idx_median"] = {k: float(np.nanmedian(v[sel])) for k, v in sc["idx"].items()}
    # share of the clear AOI that is not vegetated (NDVI < 0.3): cleared, bare or built
    sc["nonveg_pct"] = 100 * float((sc["idx"]["NDVI"][sel] < 0.3).mean())

dates = [pd.Timestamp(sc["date"]) for sc in scenes]
INDEX_COLORS = {"NDVI": "#2a78d6", "BSI": "#eb6834", "NDBI": "#1baf7a"}
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(17, 4.8))
for k, col in INDEX_COLORS.items():
    vals = [sc["idx_median"][k] for sc in scenes]
    ax.plot(dates, vals, "-o", color=col, lw=2, ms=8, label=k)
    ax.annotate(k, (dates[-1], vals[-1]), xytext=(8, 0), textcoords="offset points", va="center", color="#52514e")
ax.axhline(0, color="#999999", lw=1)
ax.set_ylabel("Median over AOI")
ax.set_title("Construction indices (enhanced, clear AOI pixels)")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.08, 1))
ax2.plot(dates, [sc["nonveg_pct"] for sc in scenes], "-o", color="#2a78d6", lw=2, ms=8)
for d_, sc in zip(dates, scenes):
    ax2.annotate(f"{sc['nonveg_pct']:.0f}%", (d_, sc["nonveg_pct"]), xytext=(0, 8), textcoords="offset points", ha="center")
ax2.set_ylim(0, 105); ax2.set_ylabel("% of clear AOI")
ax2.set_title("Non-vegetated share of the AOI (NDVI < 0.3)")
for a in (ax, ax2):
    a.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m")); a.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, n, figsize=(4.6 * n, 13), squeeze=False)
ranges = {"NDVI": (-0.2, 0.9), "BSI": (-0.4, 0.4), "NDBI": (-0.5, 0.4)}
cmaps = {"NDVI": "Greens", "BSI": "Oranges", "NDBI": "Purples"}
for i, k in enumerate(("NDVI", "BSI", "NDBI")):
    for j, sc in enumerate(scenes):
        im = axes[i, j].imshow(sc["idx"][k], cmap=cmaps[k], vmin=ranges[k][0], vmax=ranges[k][1])
        aoi_outline(axes[i, j], GUIDE_RES_M, color="black")
        axes[i, j].set_title(f"{k}  {sc['date']}"); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.8, pad=0.01)
plt.show()

## 8. Sentinel-1 radar for the same dates

Sentinel-1 is a C-band radar. It sees through cloud, and its backscatter responds to structure, roughness and moisture
rather than colour:

- **VH** (cross-polarised) is strongest from vegetation volume, so it drops sharply when a site is cleared.
- **VV** rises strongly where new buildings, steel frames and cranes form corner reflectors with the ground, and drops
  over smooth new surfaces such as fresh asphalt or concrete slabs.

Processing, per date:

1. **Search**: Sentinel-1 IW GRD scenes (dual polarisation VV + VH) within `S1_WINDOW_DAYS` of the date's Sentinel-2
   scene, from the Earth Search STAC API. If that search fails, the notebook lists the open `sentinel-s1-l1c` bucket
   instead, which is slower.
2. **One track for all dates**: the relative orbit that covers every date, with the most acquisitions. Mixing
   tracks would mix viewing angles, and backscatter changes with the angle.
3. **Read and calibrate**: only the image window over the AOI is read. It is converted to σ⁰ (sigma nought) with the
   product's calibration tables, after the thermal noise annotation is subtracted.
4. **Geocode**: the window is warped onto the 10 m grid using the product's tie points (thin-plate spline). There is no
   terrain flattening, which is fine for the usually flat terrain of a construction site.
5. **Average** each date's acquisitions (in linear power), then **align** with Sentinel-2. GRD tie-point heights can be
   tens of metres off the real terrain, and a height error moves the image sideways by error / tan(incidence angle)
   (about 60 m at the sample site). The shift is the same for every date of one track. It is estimated once, by
   correlating mean VH with mean NDVI, and applied only when it clearly improves the match.
6. **Speckle filter** (Lee, `S1_LEE_SIZE` window), conversion to dB, and bilinear upsampling to the 2 m grid for the
   combined change analysis. Radar detail stays at about 20 m (the GRD resolution), so the 2 m edges in the combined maps
   still come from the enhanced Sentinel-2.

In [ ]:
S1_BUCKET = "https://sentinel-s1-l1c.s3.amazonaws.com"
S1_GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tiff",
                   GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="2")
S1_PRODUCT_RE = re.compile(r"(S1[A-D])_IW_GRDH_1SDV_(\d{8}T\d{6})_\d{8}T\d{6}_(\d{6})_[0-9A-F]{6}_[0-9A-F]{4}", re.I)


def _http_text(url):
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            return r.text
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)


def _floats(node, tag):
    return np.array(node.find(tag).text.split(), dtype=np.float64)


def _lut_2d(vectors, value_tag, rows, cols):
    # Bilinear interpolation of an annotation LUT (vectors at given lines, samples at given pixels) on rows x cols
    lines = np.array([int(v.find("line").text) for v in vectors], dtype=np.float64)
    per_line = np.stack([np.interp(cols, _floats(v, "pixel"), _floats(v, value_tag)) for v in vectors])
    return np.stack([np.interp(rows, lines, per_line[:, k]) for k in range(len(cols))], axis=1)


def _noise_2d(xml, rows, cols):
    # Thermal noise power on the window: range LUT x azimuth LUT (IPF >= 2.9), or range LUT only (older products)
    root = ET.fromstring(xml)
    rng = root.findall(".//noiseRangeVector")
    if rng:
        noise = _lut_2d(rng, "noiseRangeLut", rows, cols)
    else:
        noise = _lut_2d(root.findall(".//noiseVector"), "noiseLut", rows, cols)
    az = root.findall(".//noiseAzimuthVector")
    if az:
        scale = np.ones_like(noise)
        for v in az:
            r0, r1 = int(v.find("firstAzimuthLine").text), int(v.find("lastAzimuthLine").text)
            c0, c1 = int(v.find("firstRangeSample").text), int(v.find("lastRangeSample").text)
            rsel = (rows >= r0) & (rows <= r1)
            csel = (cols >= c0) & (cols <= c1)
            if rsel.any() and csel.any():
                lut = np.interp(rows[rsel], _floats(v, "line"), _floats(v, "noiseAzimuthLut"))
                scale[np.ix_(rsel, csel)] = lut[:, None]
        noise = noise * scale
    return noise


def s1_product_meta(prefix):
    # Relative orbit and pass direction from the SAFE manifest (works for every Sentinel-1 unit)
    xml = _http_text(f"{prefix}/manifest.safe")
    rel = int(re.search(r'relativeOrbitNumber type="start">(\d+)<', xml).group(1))
    direction = re.search(r"<s1:pass>(\w+)</s1:pass>", xml).group(1).lower()
    return rel, direction


def s1_read_sigma0(prefix, pol, bounds, shape):
    # Calibrated, noise-corrected sigma0 (linear) on the 10 m working grid. Geocoded with the product's tie points
    # (ellipsoid + coarse terrain height from the annotation), without terrain flattening.
    url = f"/vsicurl/{prefix}/measurement/iw-{pol}.tiff"
    with rasterio.Env(**S1_GDAL_ENV), rasterio.open(url) as src:
        gcps, gcp_crs = src.gcps
        px = np.array([[g.col, g.row] for g in gcps])
        ll = np.array([[g.x, g.y] for g in gcps])
        # Window: the working grid's corners mapped to image pixels with a local affine fit on nearby tie points
        lon0, lat0 = aoi_wgs84.centroid.x, aoi_wgs84.centroid.y
        near = np.argsort(np.hypot((ll[:, 0] - lon0) * np.cos(np.radians(lat0)), ll[:, 1] - lat0))[:12]
        A = np.c_[ll[near], np.ones(len(near))]
        coef, *_ = np.linalg.lstsq(A, px[near], rcond=None)
        l, b, r, t = transform_bounds(work_crs, gcp_crs, *bounds)
        corners = np.array([[l, b, 1], [l, t, 1], [r, b, 1], [r, t, 1]]) @ coef
        c0, r0 = np.floor(corners.min(0)).astype(int) - 40
        c1, r1 = np.ceil(corners.max(0)).astype(int) + 40
        c0, r0 = max(c0, 0), max(r0, 0)
        c1, r1 = min(c1, src.width), min(r1, src.height)
        if c1 - c0 < 10 or r1 - r0 < 10:
            return None
        dn = src.read(1, window=Window(c0, r0, c1 - c0, r1 - r0)).astype(np.float64)
    rows, cols = np.arange(r0, r1, dtype=np.float64), np.arange(c0, c1, dtype=np.float64)
    cal = ET.fromstring(_http_text(f"{prefix}/annotation/calibration/calibration-iw-{pol}.xml"))
    a_sigma = _lut_2d(cal.findall(".//calibrationVector"), "sigmaNought", rows, cols)
    noise = _noise_2d(_http_text(f"{prefix}/annotation/calibration/noise-iw-{pol}.xml"), rows, cols)
    sigma0 = np.clip(dn ** 2 - noise, 0, None) / a_sigma ** 2
    sigma0 = np.maximum(sigma0, 1e-4)  # -40 dB floor where noise removal leaves nothing
    sigma0[dn == 0] = np.nan

    # Tie points relative to the window, then a thin-plate-spline warp onto the 10 m grid
    win_gcps = [GroundControlPoint(row=g.row - r0, col=g.col - c0, x=g.x, y=g.y, z=g.z) for g in gcps]
    out = np.full(shape, np.nan, dtype=np.float32)
    reproject(sigma0.astype(np.float32), out, gcps=win_gcps, src_crs=gcp_crs, src_nodata=np.nan,
              dst_transform=transform_from_bounds(*bounds, out.shape[1], out.shape[0]), dst_crs=work_crs,
              dst_nodata=np.nan, resampling=Resampling.average, SRC_METHOD="GCP_TPS")
    return out


# ---- Finding acquisitions ----
def _grid_polygon_wgs84():
    return box(*transform_bounds(work_crs, "EPSG:4326", *work_bounds))


def s1_search_stac(t_from, t_to):
    # Earth Search lists Sentinel-1 GRD scenes; the pixels are read from the open sentinel-s1-l1c bucket
    found = []
    for it in Client.open(STAC_API_URL).search(collections=["sentinel-1-grd"], intersects=_grid_polygon_wgs84().__geo_interface__,
                                               datetime=f"{t_from:%Y-%m-%d}/{t_to:%Y-%m-%d}").items():
        hrefs = [a.href for a in it.assets.values()] + [it.id]
        m = next((S1_PRODUCT_RE.search(h) for h in hrefs if S1_PRODUCT_RE.search(h)), None)
        if m is None or not shape(it.geometry).contains(_grid_polygon_wgs84()):
            continue
        t = datetime.strptime(m.group(2), "%Y%m%dT%H%M%S")
        found.append(dict(id=m.group(0), time=t, prefix=f"{S1_BUCKET}/GRD/{t.year}/{t.month}/{t.day}/IW/DV/{m.group(0)}"))
    return found


_S1_PASS_MINUTES = set()


def s1_search_bucket(t_from, t_to):
    # Fallback without the STAC API: list the bucket day by day and check the footprints of the scenes acquired
    # within 75 minutes of the local morning / evening overpass time
    # Once some passes are known, later dates only need the same times of day (one track always passes at the same time)
    lon = aoi_wgs84.centroid.x
    windows, tol = ([((h - lon / 15) * 60) % 1440 for h in (6.0, 18.0)], 75) if not _S1_PASS_MINUTES \
        else (sorted(_S1_PASS_MINUTES), 5)
    grid = _grid_polygon_wgs84()

    def footprint(p):
        js = json.loads(_http_text(f"{S1_BUCKET}/{p}productInfo.json"))
        return p, shape({"type": js["footprint"]["type"], "coordinates": js["footprint"]["coordinates"]})

    found, day = [], t_from
    while day <= t_to:
        prefixes, token = [], None
        while True:
            params = {"list-type": "2", "prefix": f"GRD/{day.year}/{day.month}/{day.day}/IW/DV/", "delimiter": "/"}
            if token:
                params["continuation-token"] = token
            xml = _http_text(requests.Request("GET", S1_BUCKET, params=params).prepare().url)
            prefixes += re.findall(r"<Prefix>(GRD/[^<]+/)</Prefix>", xml)
            m = re.search(r"<NextContinuationToken>([^<]+)<", xml)
            if not m:
                break
            token = m.group(1)
        cand = []
        for p in prefixes:
            m = S1_PRODUCT_RE.search(p)
            if m:
                t = datetime.strptime(m.group(2), "%Y%m%dT%H%M%S")
                minute = t.hour * 60 + t.minute
                if any(min(abs(minute - w), 1440 - abs(minute - w)) <= tol for w in windows):
                    cand.append(p)
        with ThreadPoolExecutor(8) as ex:
            for p, fp in ex.map(footprint, cand):
                if fp.contains(grid):
                    m = S1_PRODUCT_RE.search(p)
                    t = datetime.strptime(m.group(2), "%Y%m%dT%H%M%S")
                    _S1_PASS_MINUTES.add(t.hour * 60 + t.minute)
                    found.append(dict(id=m.group(0), time=t, prefix=f"{S1_BUCKET}/{p.rstrip('/')}"))
        day += timedelta(days=1)
    return found


def s1_candidates(center_date):
    t0 = datetime.combine(center_date, datetime.min.time())
    t_from, t_to = t0 - timedelta(days=S1_WINDOW_DAYS), t0 + timedelta(days=S1_WINDOW_DAYS)
    try:
        found = s1_search_stac(t_from, t_to)
    except Exception as e:
        print(f"   STAC search for Sentinel-1 failed ({type(e).__name__}); listing the bucket instead (slower)")
        found = s1_search_bucket(t_from, t_to)
    # One product per acquisition: reprocessed duplicates and overlapping frames of the same pass are dropped
    by_pass = {}
    for f in sorted(found, key=lambda f: f["id"]):
        by_pass.setdefault(S1_PRODUCT_RE.search(f["id"]).group(1, 3), f)  # (unit, absolute orbit)
    out = list(by_pass.values())
    for f in out:
        f["rel_orbit"], f["direction"] = s1_product_meta(f["prefix"])
    return out


# ---- Stacking, co-registration, speckle filter ----
def lee_filter(img, size, enl):
    # Lee filter on linear intensity: smooths speckle in homogeneous areas and keeps edges
    valid = np.isfinite(img)
    x = np.where(valid, img, 0.0)
    w = uniform_filter(valid.astype(float), size)
    mean = uniform_filter(x, size) / np.maximum(w, 1e-6)
    sq = uniform_filter(x * x, size) / np.maximum(w, 1e-6)
    var = np.maximum(sq - mean ** 2, 0)
    noise_var = (mean ** 2) / enl
    k = np.clip((var - noise_var) / np.maximum(var, 1e-12), 0, 1)
    out = mean + k * (x - mean)
    out[~valid] = np.nan
    return out


def estimate_shift(sar_pad, ref, pad):
    # Shift (rows, cols) of the padded SAR layer that best matches an optical reference on the working grid,
    # by correlation. Returns ((r, dy, dx), r at zero shift)
    h, w = ref.shape
    best, zero = (-2.0, 0, 0), -2.0
    for dy in range(-pad, pad + 1):
        for dx in range(-pad, pad + 1):
            a = sar_pad[pad - dy: pad - dy + h, pad - dx: pad - dx + w]
            ok = np.isfinite(a) & np.isfinite(ref)
            r = np.corrcoef(a[ok], ref[ok])[0, 1] if ok.sum() > 100 else -2.0
            if dy == 0 and dx == 0:
                zero = r
            if r > best[0]:
                best = (r, dy, dx)
    return best, zero

In [ ]:
if USE_S1:
    S1_PAD = 15   # 10 m pixels of margin read around the grid, also the largest co-registration shift tried
    pad_m = S1_PAD * S2_RES_M
    s1_bounds = BoundingBox(work_bounds.left - pad_m, work_bounds.bottom - pad_m, work_bounds.right + pad_m, work_bounds.top + pad_m)
    s1_shape = (n_rows + 2 * S1_PAD, n_cols + 2 * S1_PAD)

    print("Searching Sentinel-1 acquisitions ...")
    for sc in scenes:
        sc["s1_candidates"] = s1_candidates(sc["date"])
        by_orbit = {}
        for c in sc["s1_candidates"]:
            by_orbit.setdefault((c["rel_orbit"], c["direction"]), []).append(c["time"].strftime("%m-%d"))
        print(f"   {sc['date']}: " + ("; ".join(f"track {o} {d}: {', '.join(sorted(t))}" for (o, d), t in sorted(by_orbit.items()))
                                     or "nothing found"))

    common = set.intersection(*[{c["rel_orbit"] for c in sc["s1_candidates"]} for sc in scenes])
    if S1_ORBIT is not None:
        if S1_ORBIT not in common:
            raise RuntimeError(f"Track {S1_ORBIT} does not cover every date; tracks covering all dates: {sorted(common)}")
        s1_orbit = S1_ORBIT
    elif common:
        # most acquisitions in total, then the smallest time gap to the Sentinel-2 dates
        def score(o):
            n_acq = sum(c["rel_orbit"] == o for sc in scenes for c in sc["s1_candidates"])
            gap = sum(min(abs((c["time"].date() - sc["date"]).days) for c in sc["s1_candidates"] if c["rel_orbit"] == o)
                      for sc in scenes)
            return (-n_acq, gap)
        s1_orbit = min(common, key=score)
    else:
        raise RuntimeError("No single Sentinel-1 track covers the AOI on every date: raise S1_WINDOW_DAYS, "
                           "or set USE_S1 = False")
    s1_direction = next(c["direction"] for sc in scenes for c in sc["s1_candidates"] if c["rel_orbit"] == s1_orbit)
    print(f"Using track {s1_orbit} ({s1_direction}) for every date, so all dates share the same viewing geometry")

    for sc in scenes:
        prods = sorted((c for c in sc["s1_candidates"] if c["rel_orbit"] == s1_orbit), key=lambda c: c["time"])
        stack = {pol: [] for pol in ("vv", "vh")}
        for c in prods:
            for pol in stack:
                stack[pol].append(s1_read_sigma0(c["prefix"], pol, s1_bounds, s1_shape))
        sc["s1_pad"] = {pol: np.nanmean(np.stack(v), axis=0) for pol, v in stack.items()}
        sc["s1_ids"] = [c["id"] for c in prods]
        sc["s1_dates"] = [c["time"].date() for c in prods]
        print(f"   {sc['date']}: {len(prods)} acquisition(s) averaged: {', '.join(str(d) for d in sc['s1_dates'])}")

    # Co-registration with Sentinel-2. GRD tie points can sit tens of metres above or below the real terrain, which moves
    # the image along the range direction by (height error / tan(incidence)). The shift is the same for every date of
    # one track, so it is estimated once: mean VH (dB) against mean NDVI, both strongly driven by vegetation.
    mean_vh = 10 * np.log10(np.nanmean(np.stack([sc["s1_pad"]["vh"] for sc in scenes]), axis=0))
    mean_ndvi = np.nanmean(np.stack([(sc["s2"][..., 7] - sc["s2"][..., 3]) / (sc["s2"][..., 7] + sc["s2"][..., 3] + 1e-6)
                                     for sc in scenes]), axis=0)
    (r_best, dy, dx), r_zero = estimate_shift(mean_vh, mean_ndvi, S1_PAD)
    if r_best < 0.2 or r_best - r_zero < 0.02 or max(abs(dy), abs(dx)) == S1_PAD:
        print(f"Co-registration: no reliable shift found (correlation {r_zero:.2f} unshifted, {r_best:.2f} best); not shifting")
        dy = dx = 0
    else:
        print(f"Co-registration: moved Sentinel-1 {-dy * S2_RES_M:+.0f} m north and {dx * S2_RES_M:+.0f} m east "
              f"(VH-NDVI correlation {r_zero:.2f} -> {r_best:.2f})")
    s1_shift_px = (dy, dx)

    for sc in scenes:
        sc["s1_db"] = {}
        for pol, lin in sc["s1_pad"].items():
            lin = lin[S1_PAD - dy: S1_PAD - dy + n_rows, S1_PAD - dx: S1_PAD - dx + n_cols]
            lin = lee_filter(lin, S1_LEE_SIZE, enl=4.4 * len(sc["s1_ids"]))
            sc["s1_db"][pol] = (10 * np.log10(np.maximum(lin, 1e-4))).astype(np.float32)
        sc["s1_db_hr"] = {pol: resize(v, (n_rows * SCALE, n_cols * SCALE), order=1, mode="edge", preserve_range=True)
                          .astype(np.float32) for pol, v in sc["s1_db"].items()}
        write_geotiff(os.path.join(OUT_DIR, f"sentinel1_sigma0_db_10m_{sc['date']}.tif"),
                      np.stack([sc["s1_db"]["vv"], sc["s1_db"]["vh"]], -1), ["VV_dB", "VH_dB"])
else:
    print('USE_S1 = False: skipping Sentinel-1')

In [ ]:
if USE_S1:
    # Radar images per date (VV, VH in dB) and the AOI medians over time
    fig, axes = plt.subplots(2, n, figsize=(4.6 * n, 9.2), squeeze=False)
    for j, sc in enumerate(scenes):
        for i, pol in enumerate(("vv", "vh")):
            im = axes[i, j].imshow(sc["s1_db"][pol], cmap="gray", vmin=(-20, -26)[i], vmax=(0, -8)[i], interpolation="nearest")
            aoi_outline(axes[i, j], S2_RES_M)
            axes[i, j].set_title(f"{pol.upper()} dB  {sc['date']}\n(S1: {', '.join(d.strftime('%m-%d') for d in sc['s1_dates'])})",
                                 fontsize=10)
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
            if j == n - 1:
                fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.8, pad=0.01, label="dB")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4.2))
    for pol, col in (("vv", "#2a78d6"), ("vh", "#eb6834")):
        vals = [float(np.nanmedian(sc["s1_db"][pol][AOI_MASK_10])) for sc in scenes]
        ax.plot(dates, vals, "-o", color=col, lw=2, ms=8, label=pol.upper())
        ax.annotate(pol.upper(), (dates[-1], vals[-1]), xytext=(8, 0), textcoords="offset points", va="center", color="#52514e")
    ax.set_ylabel("Median backscatter over AOI (dB)")
    ax.set_title(f"Sentinel-1 track {s1_orbit} ({s1_direction})")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m")); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('USE_S1 = False: skipping Sentinel-1')

## 9. Change between dates

For each interval (each date to the next, plus first to last):

- **Optical change score**: the change-vector magnitude (CVA), which is the length of the 12-band reflectance difference
  per pixel, divided by its threshold. With `CVA_THRESHOLD = None` the threshold is set automatically by Otsu's method on
  the CVA values of the whole working area (AOI plus buffer), with a floor of 0.05 so that two near-identical dates do
  not produce spurious change. A score of 1 or more means optical change on its own.
- **Radar change score**: the root mean square of the VV and VH changes in dB, divided by `S1_CHANGE_DB` (3 dB by default).
- **Combined score** = (1 − `S1_WEIGHT`) × optical + `S1_WEIGHT` × radar. A pixel is changed when the combined score is
  1 or more. With equal weights:
  - a change both sensors see is kept;
  - a weak optical change with no radar support, such as a slight colour or phenology difference, is dropped;
  - a strong change seen by only one sensor (about twice its threshold) is still kept.
- **Confidence** of each changed pixel: *optical and radar* (both scores at least 1), *optical only*, or *radar only*.
- **Change type** inside the mask, checked in this order:
  - *new structure (radar)*: VV rises by more than `S1_CHANGE_DB` and the end state is not vegetated (NDVI < 0.3).
    New halls, steel frames and cranes form corner reflectors with the ground;
  - *vegetation cleared*: NDVI drops by more than `NDVI_CHANGE` from a vegetated start (NDVI > 0.3);
  - *new bright surface*: visible brightness (mean of B02, B03, B04) rises by more than 0.04 and the end state is
    not vegetated (NDVI < 0.2): new roofs and concrete, but also freshly graded light-coloured ground;
  - *earthworks / bare soil*: BSI rises by more than 0.05 and the end state is not vegetated (NDVI < 0.3);
  - *vegetation regrowth*: NDVI rises by more than `NDVI_CHANGE` to a vegetated end state (NDVI > 0.3);
  - *other change*: anything else above the threshold (for example seasonal change in the surroundings, or material
    changing on an already built surface).

Pixels that are cloudy on either date are excluded. The maps show the optical-only result next to the combined one, so
you can see what the radar adds or removes. The combined analysis is also run at 10 m, for the summary table.

There is no ground truth here, so the notebook cannot measure accuracy directly. As a rough check, it reports how much of
the surrounding buffer (outside the AOI) is flagged as changed. If that area was really stable, flagged change there is
mostly false alarm, and a method that flags less of it is more selective.

In [ ]:
CHANGE_CLASSES = ["no change", "new structure (radar)", "vegetation cleared", "new bright surface",
                  "earthworks / bare soil", "vegetation regrowth", "other change"]
CHANGE_COLORS = ["#f0efec", "#0e7c86", "#e34948", "#6250d6", "#eda100", "#008300", "#a9a8a0"]
CONF_CLASSES = ["no change", "radar only", "optical only", "optical and radar"]
CONF_COLORS = ["#f0efec", "#eda100", "#2a78d6", "#1d1d1b"]
PIXEL_HA = {GUIDE_RES_M: GUIDE_RES_M ** 2 / 1e4, S2_RES_M: S2_RES_M ** 2 / 1e4}


def change_between(a, b, res, threshold=CVA_THRESHOLD, use_s1=USE_S1):
    hr = res == GUIDE_RES_M
    ca, cb = (a["enh"], b["enh"]) if hr else (a["s2"], b["s2"])
    ia, ib = (a["idx"], b["idx"]) if hr else (a["idx_10"], b["idx_10"])
    clear = (a["clear_hr"] & b["clear_hr"]) if hr else (a["clear_10"] & b["clear_10"])
    inside = (AOI_MASK_HR if hr else AOI_MASK_10)

    cva = np.linalg.norm(cb - ca, axis=-1)
    cva[~clear] = np.nan
    if threshold is None:
        vals = cva[np.isfinite(cva)]
        threshold = max(float(threshold_otsu(vals)), 0.05)
    z_opt = cva / threshold
    if use_s1:
        sa, sb = (a["s1_db_hr"], b["s1_db_hr"]) if hr else (a["s1_db"], b["s1_db"])
        d_vv, d_vh = sb["vv"] - sa["vv"], sb["vh"] - sa["vh"]
        z_sar = np.sqrt((d_vv ** 2 + d_vh ** 2) / 2) / S1_CHANGE_DB
        z_sar[~np.isfinite(z_sar)] = 0.0
        score = (1 - S1_WEIGHT) * z_opt + S1_WEIGHT * z_sar
    else:
        d_vv = np.zeros(cva.shape, np.float32)
        z_sar = np.zeros(cva.shape, np.float32)
        score = z_opt
    changed = np.isfinite(cva) & (score >= 1)
    conf = np.where(changed, 1 * (z_sar >= 1) + 2 * (z_opt >= 1), 0).astype(np.uint8)
    conf[changed & (conf == 0)] = 3   # neither alone, but the combined evidence passes (only with uneven weights)

    d_ndvi, d_bsi = ib["NDVI"] - ia["NDVI"], ib["BSI"] - ia["BSI"]
    d_bright = cb[..., 1:4].mean(-1) - ca[..., 1:4].mean(-1)   # B02, B03, B04
    rules = [  # (class index, condition), first match wins
        (1, (d_vv > S1_CHANGE_DB) & (ib["NDVI"] < 0.3)),
        (2, (d_ndvi < -NDVI_CHANGE) & (ia["NDVI"] > 0.3)),
        (3, (d_bright > 0.04) & (ib["NDVI"] < 0.2)),
        (4, (d_bsi > 0.05) & (ib["NDVI"] < 0.3)),
        (5, (d_ndvi > NDVI_CHANGE) & (ib["NDVI"] > 0.3)),
        (6, np.ones(cva.shape, bool)),
    ]
    cls = np.zeros(cva.shape, np.uint8)
    todo = changed.copy()
    for c, cond in rules:
        hit = todo & cond
        cls[hit] = c
        todo &= ~hit
    cls[~clear] = 255
    conf[~clear] = 255

    valid = inside & clear
    outside = ~inside & clear
    area = {CHANGE_CLASSES[c]: float(((cls == c) & valid).sum() * PIXEL_HA[res]) for c in range(1, len(CHANGE_CLASSES))}
    conf_ha = {CONF_CLASSES[c]: float(((conf == c) & valid).sum() * PIXEL_HA[res]) for c in range(1, len(CONF_CLASSES))}
    return dict(cva=cva, cls=cls, conf=conf, z_sar=z_sar, threshold=threshold, area_ha=area, conf_ha=conf_ha,
                changed_ha=sum(area.values()), changed_pct=100 * float((changed & valid).sum()) / max(valid.sum(), 1),
                buffer_changed_pct=100 * float((changed & outside).sum()) / max(outside.sum(), 1),
                aoi_valid_ha=float(valid.sum() * PIXEL_HA[res]))


pairs = [(i, i + 1) for i in range(n - 1)]
if n > 2:
    pairs.append((0, n - 1))
changes = []
for i, j in pairs:
    hr = change_between(scenes[i], scenes[j], GUIDE_RES_M)
    lr = change_between(scenes[i], scenes[j], S2_RES_M, threshold=hr["threshold"])  # same threshold at both scales
    opt = change_between(scenes[i], scenes[j], GUIDE_RES_M, threshold=hr["threshold"], use_s1=False) if USE_S1 else hr
    label = f"{scenes[i]['date']} -> {scenes[j]['date']}"
    changes.append(dict(i=i, j=j, label=label, hr=hr, lr=lr, opt=opt))
    tag = f"{scenes[i]['date']}_{scenes[j]['date']}"
    write_geotiff(os.path.join(OUT_DIR, f"change_cva_{tag}.tif"), hr["cva"][..., None], ["CVA"])
    write_geotiff(os.path.join(OUT_DIR, f"change_class_{tag}.tif"), hr["cls"][..., None], ["class"], dtype="uint8")
    if USE_S1:
        write_geotiff(os.path.join(OUT_DIR, f"change_confidence_{tag}.tif"), hr["conf"][..., None], ["confidence"], dtype="uint8")
        print(f"{label}: optical threshold {hr['threshold']:.3f}; changed in AOI: {opt['changed_ha']:.1f} ha optical only "
              f"-> {hr['changed_ha']:.1f} ha combined ({hr['changed_pct']:.0f}% of clear AOI; "
              f"{hr['conf_ha']['optical and radar']:.1f} ha seen by both sensors). "
              f"Flagged in the buffer outside the AOI: {opt['buffer_changed_pct']:.1f}% -> {hr['buffer_changed_pct']:.1f}%")
    else:
        print(f"{label}: threshold {hr['threshold']:.3f}, changed {hr['changed_ha']:.1f} ha "
              f"({hr['changed_pct']:.0f}% of clear AOI) at {GUIDE_RES_M:g} m; {lr['changed_ha']:.1f} ha at {S2_RES_M:g} m")

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

cls_cmap = ListedColormap(CHANGE_COLORS)


def show_classes(ax, cls):
    img = np.ma.masked_equal(cls, 255)
    ax.imshow(img, cmap=cls_cmap, vmin=0, vmax=len(CHANGE_CLASSES) - 1, interpolation="nearest")


for ch in changes:
    a, b = scenes[ch["i"]], scenes[ch["j"]]
    fig, axes = plt.subplots(2, 3, figsize=(17, 12.2))
    axes = axes.ravel()
    axes[0].imshow(stretch(bands(a["enh"], TRUE_COLOR), LIMITS)); axes[0].set_title(f"{a['date']} enhanced")
    axes[1].imshow(stretch(bands(b["enh"], TRUE_COLOR), LIMITS)); axes[1].set_title(f"{b['date']} enhanced")
    if USE_S1:
        im = axes[2].imshow(ch["hr"]["z_sar"] * S1_CHANGE_DB, cmap="magma", vmin=0, vmax=3 * S1_CHANGE_DB)
        fig.colorbar(im, ax=axes[2], shrink=0.75, label="dB")
        axes[2].set_title("Radar change (RMS of VV and VH change)")
        show_classes(axes[3], ch["opt"]["cls"]); axes[3].set_title(f"Change, Sentinel-2 only ({GUIDE_RES_M:g} m)")
        show_classes(axes[4], ch["hr"]["cls"]); axes[4].set_title("Change, Sentinel-2 + Sentinel-1")
        conf = np.ma.masked_equal(ch["hr"]["conf"], 255)
        axes[5].imshow(conf, cmap=ListedColormap(CONF_COLORS), vmin=0, vmax=3, interpolation="nearest")
        axes[5].set_title("Which sensor sees the change")
        axes[5].legend(handles=[Patch(color=c, label=l) for c, l in zip(CONF_COLORS[1:], CONF_CLASSES[1:])],
                       loc="upper left", bbox_to_anchor=(0, -0.02), ncol=3, frameon=False, fontsize=10)
    else:
        show_classes(axes[2], ch["hr"]["cls"]); axes[2].set_title(f"Change at {GUIDE_RES_M:g} m (enhanced)")
        show_classes(axes[3], np.repeat(np.repeat(ch["lr"]["cls"], SCALE, 0), SCALE, 1))
        axes[3].set_title(f"Change at {S2_RES_M:g} m")
        axes[4].axis("off"); axes[5].axis("off")
    for k, ax in enumerate(axes):
        if ax.has_data():
            aoi_outline(ax, GUIDE_RES_M, color="yellow" if k < 3 else "black")
        ax.set_xticks([]); ax.set_yticks([])
    fig.legend(handles=[Patch(color=c, label=l) for c, l in zip(CHANGE_COLORS, CHANGE_CLASSES)] +
               [Patch(facecolor="white", edgecolor="#999999", label="cloud / no data")],
               loc="lower center", ncol=4, frameon=False)
    fig.suptitle(ch["label"], fontweight="bold")
    plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.show()

### Changed area per interval and summary table

In [ ]:
consecutive = [ch for ch in changes if ch["j"] == ch["i"] + 1]
x = np.arange(len(consecutive))
fig, ax = plt.subplots(figsize=(max(7, 3 * len(consecutive)), 4.8))
bottom = np.zeros(len(consecutive))
for c in range(1, len(CHANGE_CLASSES)):
    vals = np.array([ch["hr"]["area_ha"][CHANGE_CLASSES[c]] for ch in consecutive])
    ax.bar(x, vals, bottom=bottom, width=0.55, color=CHANGE_COLORS[c], label=CHANGE_CLASSES[c],
           edgecolor="white", linewidth=2)
    bottom += vals
for xi, total in zip(x, bottom):
    ax.annotate(f"{total:.1f} ha", (xi, total), xytext=(0, 4), textcoords="offset points", ha="center")
ax.set_xticks(x); ax.set_xticklabels([ch["label"].replace(" -> ", "\nto ") for ch in consecutive])
ax.set_ylim(0, bottom.max() * 1.15 + 1e-6)
ax.set_ylabel("Changed area inside AOI (ha)")
ax.set_title(f"Change per interval (enhanced {GUIDE_RES_M:g} m" + (" + Sentinel-1)" if USE_S1 else ")"))
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1)); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

rows = []
for k, sc in enumerate(scenes):
    prev = next((ch for ch in consecutive if ch["j"] == k), None)
    rows.append({
        "requested_date": sc["requested"], "scene_date": str(sc["date"]), "scene_id": sc["item"].id,
        "aoi_clear_pct": round(100 * sc["clear_frac"], 1),
        **{f"{name}_median": round(v, 3) for name, v in sc["idx_median"].items()},
        "non_vegetated_pct": round(sc["nonveg_pct"], 1),
        "SAM_vs_10m_deg": round(sc["consistency"]["SAM"], 2), "ERGAS_vs_10m": round(sc["consistency"]["ERGAS"], 2),
        "changed_since_prev_ha": round(prev["hr"]["changed_ha"], 2) if prev else None,
        "changed_since_prev_ha_10m": round(prev["lr"]["changed_ha"], 2) if prev else None,
    })
    if USE_S1:
        rows[-1].update({
            "s1_track": s1_orbit, "s1_dates": " ".join(str(d) for d in sc["s1_dates"]),
            "VV_median_dB": round(float(np.nanmedian(sc["s1_db"]["vv"][AOI_MASK_10])), 2),
            "VH_median_dB": round(float(np.nanmedian(sc["s1_db"]["vh"][AOI_MASK_10])), 2),
            "changed_since_prev_ha_optical_only": round(prev["opt"]["changed_ha"], 2) if prev else None,
            "changed_since_prev_ha_both_sensors": round(prev["hr"]["conf_ha"]["optical and radar"], 2) if prev else None,
        })
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT_DIR, "monitoring_summary.csv"), index=False)
if n > 2:
    total = changes[-1]["hr"]
    print(f"First to last date ({changes[-1]['label']}): {total['changed_ha']:.1f} ha changed "
          f"({total['changed_pct']:.0f}% of the clear AOI): " +
          ", ".join(f"{k} {v:.1f} ha" for k, v in total["area_ha"].items()))
summary

## 10. Optional: reduced-resolution check of the method (Wald protocol)

There is no real 2 m Sentinel-2 image to compare with, so this tests the method where the truth is known. The first
date is degraded to 50 m, enhanced back to 10 m with its own 10 m RGB as the guide, and compared with the real 10 m bands.
The comparison includes plain bicubic upsampling and the same enhancement with the sparse residual switched off, which
shows what the patch stage adds. Set `RUN_WALD_CHECK = True` to run it. It takes about as long as enhancing one date.

For reference, these are the reduced-resolution results used to pick the defaults (`DETAIL_WEIGHT = 0.25`,
`GUIDE_RADIUS = 4`, `STRIDE = 2`):

| Scene | Bicubic RMSE / ERGAS | Residual off | `RESIDUAL_WEIGHT = 0.5` |
|---|---|---|---|
| Marienkas, Namibia, 2024-07-16 | 0.0211 / 2.88 | 0.0158 / 2.12 | **0.0150 / 2.01** |
| Giga Berlin, 2020-09-20 | 0.0184 / 3.17 | 0.0150 / 2.38 | **0.0142 / 2.26** |
| Giga Berlin, 2021-04-21 | 0.0196 / 2.93 | 0.0157 / 2.21 | **0.0150 / 2.09** |
| Giga Berlin, 2020-03-27 | 0.0125 / 2.47 | 0.0100 / 1.83 | 0.0100 / 1.85 |

In [ ]:
RUN_WALD_CHECK = False

if RUN_WALD_CHECK:
    f = SCALE
    s2_0 = scenes[0]["s2"]
    h, w = (s2_0.shape[0] // f) * f, (s2_0.shape[1] // f) * f
    truth = s2_0[:h, :w]
    low = block_mean(truth, f)
    rgb = bands(truth, TRUE_COLOR)
    lo, hi = np.nanpercentile(rgb, 2, axis=(0, 1)), np.nanpercentile(rgb, 98, axis=(0, 1))
    guide = (np.clip((rgb - lo) / (hi - lo), 0, 1) * 254 + 1).astype(np.uint8)
    extent = (work_bounds.left, work_bounds.top - h * S2_RES_M, work_bounds.left + w * S2_RES_M, work_bounds.top)
    for path, arr, dt in (("wald_s2_low.tif", low, "float32"), ("wald_guide_rgb.tif", guide, "uint8")):
        tr = transform_from_bounds(*extent, arr.shape[1], arr.shape[0])
        with rasterio.open(os.path.join(OUT_DIR, path), "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1],
                           count=arr.shape[2], dtype=dt, crs=work_crs, transform=tr) as dst:
            dst.write(np.moveaxis(arr, -1, 0).astype(dt))
    wald = HSIEnhancer(os.path.join(OUT_DIR, "wald_guide_rgb.tif"), os.path.join(OUT_DIR, "wald_s2_low.tif"))
    candidates = {"Bicubic": resize(low, truth.shape, order=3, mode="edge", preserve_range=True).astype(np.float32)}
    for label, rw in (("P2SR, residual off", 0.0), (f"P2SR, residual {RESIDUAL_WEIGHT:g}", RESIDUAL_WEIGHT)):
        candidates[label] = wald.fuse_to_enhance(patch_size=PATCH_SIZE, stride=STRIDE, guide_radius=GUIDE_RADIUS,
                                                 detail_weight=DETAIL_WEIGHT, residual_weight=rw)
    print(f"{'Method':<22}{'RMSE':>9}{'PSNR dB':>9}{'SAM deg':>9}{'ERGAS':>8}{'CC':>8}")
    for name, est in candidates.items():
        r = quality_metrics(truth, est, f)
        print(f"{name:<22}{r['RMSE']:>9.4f}{r['PSNR']:>9.2f}{r['SAM']:>9.2f}{r['ERGAS']:>8.2f}{r['CC']:>8.3f}")

## 11. Outputs and notes

Files in `s2_monitoring_outputs/`:

| File | Content |
|---|---|
| `sentinel2_10m_<date>.tif` | 12-band Sentinel-2 L2A reflectance on the 10 m grid, per date |
| `sentinel2_enhanced_2m_<date>.tif` | enhanced 12-band reflectance at 2 m, per date (band names in the band descriptions) |
| `guide_<source>_2m.tif` | high-resolution RGB guide (source and attribution in the GeoTIFF tags) |
| `change_cva_<date1>_<date2>.tif` | change-vector magnitude at 2 m |
| `sentinel1_sigma0_db_10m_<date>.tif` | Sentinel-1 VV and VH backscatter (dB) on the 10 m grid, per date (average of the track's acquisitions near that date, aligned and speckle filtered) |
| `change_class_<date1>_<date2>.tif` | change classes at 2 m: 0 none, 1 new structure (radar), 2 vegetation cleared, 3 new bright surface, 4 earthworks / bare soil, 5 vegetation regrowth, 6 other, 255 cloud |
| `change_confidence_<date1>_<date2>.tif` | which sensor sees the change: 0 none, 1 radar only, 2 optical only, 3 both, 255 cloud |
| `monitoring_summary.csv` | per date: scene, AOI clear %, index medians, Sentinel-1 track, dates and medians, consistency with 10 m, changed area since the previous date (combined, optical only, seen by both) |

**Tuning and interpretation**

- `RESIDUAL_WEIGHT` scales the patch-wise residual. At 1.0 it gave similar or slightly worse scores than 0.5 in the
  reduced-resolution checks, because detail injection already adds part of the same detail.
- Raise `CVA_THRESHOLD` (for example 0.08) if stable areas show up as change, or lower it to catch subtle change.
- Radar: raise `S1_WEIGHT` to trust the radar more, for example over hazy or partly cloudy dates. Lower it, or raise
  `S1_CHANGE_DB`, if rain or soil-moisture differences between dates show up as radar-only change on bare fields: wet
  soil can raise VV by several dB. Radar-only pixels are about 20 m resolution features drawn on the 2 m grid.
- The radar is not terrain flattened and not corrected for the slope of the ground, which is fine on flat sites. In
  steep terrain, use a terrain-corrected (RTC) Sentinel-1 product instead.
- Compare dates from the same season where you can; phenology alone moves NDVI and can be counted as vegetation change.
- `DETAIL_WEIGHT` sets how much guide texture is injected. Keep it low for monitoring so the static guide does not
  dominate; the change maps are driven by Sentinel-2 spectra either way, because the same guide is used for all dates.
- For a guide closer in time to each date, ESRI's World Imagery Wayback archive holds dated past versions of the
  basemap; swapping it in per date would sharpen new structures, at the cost of a different download routine.
- Check the terms of use of the guide imagery (ESRI, Google or your own source) before redistributing derived products.
  The Google option in particular is limited by the Google Maps Platform terms (see section 5).
- Try both basemaps: ESRI and Google often use different acquisition dates, so one may already show structures the
  other does not.

**Citation**: Naik, P., Chakraborty, R., Thiele, S., & Gloaguen, R. (2025). Scalable Hyperspectral Enhancement via
Patch-Wise Sparse Residual Learning: Insights from Super-Resolved EnMAP Data. *Remote Sensing*, 17(11), 1878.
https://doi.org/10.3390/rs17111878